### Connection MYSQL

In [22]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy import text

In [23]:
load_dotenv()

DB_USER = os.getenv("DB_USER");
DB_PASSWORD = os.getenv("DB_PASSWORD");
DB_HOST = os.getenv("DB_HOST");
DB_PORT = os.getenv("DB_PORT", 3306);
DB_SIAKAD = os.getenv("DB_SIAKAD");

siakadConnection = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_SIAKAD}"
siakadEngine = create_engine(siakadConnection)

### Library

In [24]:
import pandas as pd

### Dosen Kelas Laporan

In [25]:
query = text("""
SELECT
	kl.semester_code,
	semesters.name AS semester,
	semesters.year_of_study,
	matakuliah.name AS matakuliah,
	matakuliah.sks_count AS sks,
	prodi.name AS program_study,
	dosen.name AS dosen,
	pd.name AS dosen_program_study
FROM kelas_laporan as kl
	JOIN matakuliah ON kl.matkul_uuid = matakuliah.uuid
	JOIN prodi ON kl.prodi_uuid = prodi.uuid
	JOIN semesters ON kl.semester_code = semesters.code
	JOIN dosen_kelas_laporan AS dkl ON dkl.kelas_uuid = kl.uuid
	JOIN dosen_list AS dosen ON dkl.dosen_uuid = dosen.uuid
	JOIN prodi AS pd ON dosen.prodi_uuid = pd.uuid
WHERE
	kl.deleted_at IS NULL AND
	matakuliah.deleted_at IS NULL AND
	dkl.deleted_at IS NULL AND
	dosen.deleted_at IS NULL
""")

df = pd.read_sql(query, siakadEngine)

In [26]:
(
    df
    .drop_duplicates()
    .to_csv('data/dosen/dosen-kelas-laporan.csv', index=False)
)